# Global Forces

Notebook to generate the timeseries of building's global forces, generated from its pressure measurements.

## Input Variables

In [1]:
# Path for timeseries of body's pressure
body_xdmf = "/home/ubuntu/Documents/Codigos/AeroSim/cfdmod/fixtures/tests/pressure/xdmf/bodies.building.xdmf"
body_h5 = "/home/ubuntu/Documents/Codigos/AeroSim/cfdmod/fixtures/tests/pressure/xdmf/bodies.building.h5"

# Path for atmosphere probe (leve as None in case it wasn't exported)
atm_xdmf = "/home/ubuntu/Documents/Codigos/AeroSim/cfdmod/fixtures/tests/pressure/xdmf/points.point0.xdmf"
atm_h5 = "/home/ubuntu/Documents/Codigos/AeroSim/cfdmod/fixtures/tests/pressure/xdmf/points.point0.h5"

# Path to save CSV with global moments and forces
export_csv = "/home/ubuntu/Documents/Codigos/AeroSim/cfdmod/fixtures/tests/pressure/xdmf/forces.csv"

# Reference values to use for Cp
reference_velocity = 10 # m/s
fluid_density = 1.225

# Lever for MZ
lever_position = (0, 0, 0)
# Nominal area for Cf
nominal_area = 3000
# Nominal volume for Cm
nominal_volume = 30000

## Functions

In [2]:
import pathlib
import h5py
import numpy as np
import pandas as pd
from dataclasses import dataclass

def add_cp2xdmf(
    *,
    body_h5: pathlib.Path,
    atm_probe_h5: pathlib.Path | None,
    reference_vel: float,
    fluid_density: float,
):
    """Add pressure coefficient (Cp) to H5 compatible with XDMF format

    Args:
        body_h5 (pathlib.Path): path to .h5 file for body's pressure time series
        atm_probe_h5 (pathlib.Path | None): path to .h5 file for atmospheric pressure probe.
            If None consider constant atmospheric pressure of 0.
        reference_vel (float): reference velocity to use
        fluid_density (float): fluid density to use
    """

    with h5py.File(body_h5, mode="a") as f_body:
        grp_abs = f_body["pressure"]
        grp_cp = f_body.require_group("cp")
        keys = list(grp_abs.keys())

        if atm_probe_h5 is None:
            for k in keys:
                p_body = grp_abs[k]
                cp = (p_body) / (0.5 * fluid_density * reference_vel**2)
                if k in grp_cp:
                    del grp_cp[k]
                grp_cp[k] = cp
            return

        with h5py.File(atm_probe_h5) as f_atm:
            grp_atm = f_atm[f"pressure"]
            for k in keys:
                p_body = grp_abs[k]
                p_ref = grp_atm[k][0]
                cp = (p_body - p_ref) / (0.5 * fluid_density * reference_vel**2)
                if k in grp_cp:
                    del grp_cp[k]
                grp_cp[k] = cp


@dataclass
class GeomTriangles:
    """Class to represent a triangulated geometry"""
    triangles: np.ndarray
    centers: np.ndarray
    areas: np.ndarray
    normals: np.ndarray

def geometry_from_h5(body_h5: pathlib.Path) -> GeomTriangles:
    """Get the geometry from body's .h5 file"""

    with h5py.File(body_h5, mode="r") as f_body:
        triangles_idx = np.array(f_body["Triangles"]).flatten()
        vertices = np.array(f_body["Geometry"])

        triangles = vertices[triangles_idx]
        triangles = triangles.reshape(f_body["Triangles"].shape + (3, ))

    U = triangles[:, 1, :] - triangles[:, 0, :]
    V = triangles[:, 2, :] - triangles[:, 0, :]
    cross_prod = np.cross(U, V)
    areas = np.linalg.norm(cross_prod, axis=1) / 2
    normals = cross_prod / np.linalg.norm(cross_prod, axis=1)[:, np.newaxis]
    centers = triangles.mean(axis=1)
    return GeomTriangles(triangles=triangles, areas=areas, normals=normals, centers=centers)


def get_global_timeseries_forces(*, body_h5: pathlib.Path, geometry: GeomTriangles, nominal_area: float, nominal_volume: float) -> pd.DataFrame:
    """Calculate global forces on building from pressure coefficients and geometry"""

    lever_dist = geometry.centers - np.array(lever_position)
    lever = np.linalg.norm(lever_dist, axis=1)
    nx, ny = geometry.normals[:, 0], geometry.normals[:, 1]
    df_dct = {"time_step": [], "fx": [], "fy": [], "mz": []}

    with h5py.File(body_h5) as f_body:
        grp_cp = f_body["cp"]
        keys = list(grp_cp.keys())

        for k in keys:
            # key in format of (t{time_value})
            time_step = float(k[1:])
            cp = grp_cp[k] * geometry.areas
            # Force and moments by triangle
            fx = cp * nx / nominal_area
            fy = cp * ny / nominal_area
            mz = (cp * ny - cp * nx) * lever / nominal_volume
            # Append to global dictionary
            df_dct["fx"].append(fx.sum())
            df_dct["fy"].append(fy.sum())
            df_dct["mz"].append(mz.sum())
            df_dct["time_step"].append(time_step)

    return pd.DataFrame(df_dct)


## Code Execution

In [3]:
# First add pressure coeficient to H5 of XDMF
add_cp2xdmf(body_h5=body_h5, atm_probe_h5=atm_h5, reference_vel=reference_velocity, fluid_density=fluid_density)

# NOTE: the geometry can be built from a STL or any other format, as long as the triangles are the 
# same (they may be scaled, translated, rotated, etc.)
geometry = geometry_from_h5(body_h5=body_h5)

# Build an dataframe with the global forces and moments
df = get_global_timeseries_forces(body_h5=body_h5, geometry=geometry, nominal_area=nominal_area, nominal_volume=nominal_volume)

# Save data to CSV
df.to_csv(export_csv)

df

,time_step,fx,fy,mz
0,125.157166,-6.474822,0.471620,61.648022
1,125.257286,-6.486102,0.482922,61.476292
2,125.357414,-6.495677,0.488033,61.477392
3,125.457542,-6.499244,0.475274,61.337347
4,125.557663,-6.520759,0.441243,61.049171
...,...,...,...,...
94,134.568985,-7.294603,-0.335134,65.198287
95,134.669113,-7.301932,-0.356809,65.205031
96,134.769226,-7.315357,-0.375512,65.146781
97,134.869354,-7.380976,-0.422631,65.482864
